# Lab 1
## Dynamic model of a KUKA KR5 manipulator

## Steps

### 0. Import the necessary libraries

In [ ]:
from math import pi
import numpy as np
import roboticstoolbox as rtb
import matplotlib.pyplot as plt

### 1. Import robot model (ex. KUKA KR5)

In [ ]:
# Load KUKA KR5 robot model
robot = rtb.models.DH.KR5()

# This command prints the robot description and the DH parameter table
print("KUKA KR5 Robot Model and DH Parameters:")
print(robot)

### 2. Fill in the parameters of the robot model

0. Using dyn() you can view the dynamic parameters of each link

In [ ]:
# Displaying the initial dynamic parameters for all links of the KUKA KR5
for i, link in enumerate(robot.links):
    print(f"Link {i+1}:")
    print(link.dyn())
    print()

1. Let's define the mass of the links

In [ ]:
# Specifying the masses of the links (kg)
robot.links[0].m = 5.0
robot.links[1].m = 6.0
robot.links[2].m = 4.0
robot.links[3].m = 2.0
robot.links[4].m = 2.0
robot.links[5].m = 1.0
print("Link masses assigned.")

2. Let's define the center of mass of each link

In [ ]:
# Specifying positions of centers of mass [x, y, z]
robot.links[0].r = [0, 0, 0.1]
robot.links[1].r = [-0.2, 0, 0.1]
robot.links[2].r = [-0.1, 0, 0.05]
robot.links[3].r = [0, 0, 0.02]
robot.links[4].r = [0, 0, 0.02]
robot.links[5].r = [0, 0, 0.01]
print("Centers of mass assigned.")

3. Let's define the inertia tensor of each link. Filling: [Lxx, Lyy, Lzz, Lxy, Lyz, Lxz]

In [ ]:
# Specifying tensors of inertia [Ixx Iyy Izz Ixy Iyz Ixz]
for i in range(6):
    robot.links[i].I = [0.05, 0.05, 0.02, 0, 0, 0]
print("Inertia tensors assigned.")

4. Let's set the moment of inertia of the drive

In [ ]:
## Specifying moments of inertia of the drives
for i in range(6):
    robot.links[i].Jm = 1e-4
print("Motor inertia assigned.")

5. Let's define the coefficient of viscous friction of the drive

In [ ]:
# Specifying coefficients of viscous friction
for i in range(6):
    robot.links[i].B = 0.05
print("Viscous friction coefficients assigned.")

6. Let's define the coefficient of Coulomb friction of the drive

In [ ]:
# Specifying coefficients of Coulomb friction [positive, negative]
for i in range(6):
    robot.links[i].Tc = [0.1, -0.1]
print("Coulomb friction coefficients assigned.")

7. Let's set the gear ratio for each link

In [ ]:
# Specify the constraints on the generalized coordinates (Joint Limits)
robot.links[0].qlim = [-2.79, 2.79]
robot.links[1].qlim = [-0.78, 3.92]
robot.links[2].qlim = [-3.92, 0.78]
robot.links[3].qlim = [-1.91, 2.96]
robot.links[4].qlim = [-1.74, 1.74]
robot.links[5].qlim = [-4.64, 4.64]

for i in range(6):
    print(f"Joint {i+1} limits set to: {robot.links[i].qlim}")

### 3. Set the initial and final positions of the robot and plot them

In [ ]:
# Initial configuration
q_start = [0, 0, 0, 0, 0, 0]
robot.plot(q_start)
plt.show()

In [ ]:
# Final configuration
q_end = [pi/4, -pi/3, -pi/4, pi/3, -pi/3, pi/4]
robot.plot(q_end)
plt.show()

### 4. Plan the trajectory with prebuilt functions

In [ ]:
# Plan a trajectory for 2 seconds with 50 time steps
time = np.linspace(0, 2, 50)
traj = rtb.jtraj(q_start, q_end, time)

# Extract joint positions, velocities, and accelerations
q = traj.q      # Position
qd = traj.qd    # Velocity
qdd = traj.qdd  # Acceleration

print("Trajectory planning complete.")

### 5. Solve the inverse dynamics

In [ ]:
# Scenario 1: Non-zero velocities and accelerations (Full Dynamics)
tau_dynamic = robot.rne(q, qd, qdd)

# Scenario 2: Quasi-statics (Accelerations are negligible/zero)
tau_quasi = robot.rne(q, qd, qdd * 0)

# Scenario 3: Maintaining position (Velocities and accelerations are zero)
tau_static = robot.rne(q, qd * 0, qdd * 0)

print("Inverse dynamics solved for all three scenarios.")

### 6. Obtain the components of the dynamic equation

In [ ]:
# Determine numerical values at the midpoint of the trajectory
mid_index = len(time) // 2
q_mid = q[mid_index]
qd_mid = qd[mid_index]

M = robot.inertia(q_mid)         # Mass/Inertia Matrix
C = robot.coriolis(q_mid, qd_mid) # Coriolis Matrix
G = robot.gravload(q_mid)        # Gravity Vector

print(f"Numerical values at t = {time[mid_index]}s:")
print("\nMatrix M(q):\n", M)
print("\nMatrix C(q, q_dot):\n", C)
print("\nVector G(q):\n", G)

In [ ]:
# Create the figure for the Joint Moments (Torques)
plt.figure(figsize=(12, 8), dpi=100)

for g in range(6):
    plt.subplot(2, 3, g + 1)
    # Adjust spacing to match the template style
    plt.subplots_adjust(left=0.1, bottom=0.1, right=0.9, top=0.9, wspace=0.3, hspace=0.4)
    
    # Plotting the three scenarios calculated in Step 5
    plt.plot(time, tau_dynamic[:, g], linewidth=2, label=r"$\tau$")
    plt.plot(time, tau_quasi[:, g], linewidth=2, linestyle='--', label=r"$\tau_{0}$")
    plt.plot(time, tau_static[:, g], linewidth=1.5, linestyle=':', label=r"$\tau_{s}$")

    plt.title(f"Link moment {g+1}", fontsize=10)
    plt.ylabel(r"$N \cdot m$", fontsize=9)
    plt.xlabel(r"$sec$", fontsize=9)
    plt.grid(True)
    plt.legend()
    
    # Background styling to match template
    ax = plt.gca()
    ax.set_facecolor((1, 1, 1))

plt.show()